# Feature Selection 
Shows feature elimination decisions

`results/feature_selection.csv`:
- Logged results in a table

`src/features/transformers.py`:
- Custom transformers implementing engineered features (e.g., lag
features, ratios, domain-driven encodings).

`src/features/preprocessing.py`:
- Main preprocessing pipeline that drops unused features, integrates
engineered features, and can be reused in modeling.


### Description of merging procedure
##### Groundwater nitrate data cleaning (`groundwater/gwater_clean.ipynb`)

### Description of Merging Procedure

We combined groundwater nitrate measurements with environmental and agricultural datasets to create a single dataset for machine learning. The groundwater dataset served as our starting point, containing **5,280 measurements from 1,993 wells** collected between 2022 and 2025.

First, we linked **soil (SSURGO) and terrain (DEM)** features to groundwater observations using their unique well IDs. We then added **PRISM climate variables** matched to each sampling observation, accounting for the timing of nitrate measurements.

Finally, we added **historical agricultural features** calculated within 500 m and 1,000 m buffers around each well. Crop data were matched by well ID and the year preceding each nitrate measurement (e.g., nitrate from 2023 was matched with crop data from 2022).

We used **left joins** to preserve all original groundwater observations and checked for duplicate records and unmatched data. All 5,280 observations were matched to an agricultural well-year record, although some individual features have missing values due to limited spatial coverage.

The final integrated dataset contains **5,280 observations and 147 columns**, including nitrate concentrations, environmental predictors, agricultural features, and supporting metadata.

### Preliminary Feature Assessment

Our updated integrated dataset contains **5,280 groundwater nitrate observations from 1,993 wells and 147 columns**, combining soil properties, terrain, climate, and historical agricultural land-use features. Agricultural predictors are linked to nitrate measurements using crop maps from the preceding year (2021–2024 for nitrate observations from 2022–2025). This one-year lag is a retrospective modeling assumption, not an established groundwater nitrate transport time.

Preliminary EDA identified consistently positive Spearman correlations between nitrate and **crop cycles (ρ = 0.350–0.390), multicrop area (ρ = 0.323–0.369), and vegetable/berry cultivation (ρ = 0.294–0.346)** across sampling years. However, agricultural predictors are strongly correlated with one another (e.g., crop cycles and multicrop area, ρ ≈ 0.90) and with geographic characteristics such as elevation and slope. These associations may reflect overlapping environmental and spatial patterns rather than independent effects.

We will retain agricultural intensity, crop composition, soils, terrain, and climate as **candidate predictors** while excluding identifiers and inappropriate metadata from model inputs. Feature screening should be performed using training data only, with evaluation strategies accounting for repeated wells and spatial or temporal dependence. Highly correlated variables should be assessed for their additional predictive value rather than automatically removed. Final feature selection remains pending model evaluation.

#### Variables inside `groundwater_ml_integration_lag1_2021_2024_with_climate.csv`:

| Target  | Description |
|-------|-----|
| `nitrate_mg_L`		|			Nitrate concentration (mg/L) at specific geographic location and time |
|-------|-----|
| **Feature**  | ----- |
|-------|-----|
|`slopegradwta`			|		Weighted mean by intersected 500 m buffer area. |
|`brockdepmin`			|		Area-weighted depth to bedrock within 500 m. |
|`aws0150wta`				|	Area-weighted available water storage in 500 m. |
|`drclassdcd`				|	Convert drainage classes to mapped-area percentages. |
|`hydgrpdcd`			|	Convert hydrologic group classes to mapped-area percentages. |
|`soil_coverage_pct_500m`		|	Fraction of buffer with valid mapped soil data. |
|`elevation_m`				|	Sample existing 10 m DEM at well coordinates. |
|`distance_to_river_m`		|		Nearest mapped main-channel line; candidate stream order >3. |
|`topographic_position`			|	Well elevation minus mean DEM elevation within 500 m. |
|`pct_veg_berry_500m_2023`		|	Percent land 500m of the well w/ veggies, berries (2023) |
|`pct_grain_field_pasture_500m_2023`		|Percent land 500m of the well w/ grain, hay, pasture (2023) |
|`pct_idle_fallow_500m_2023`		|	Percent land 500m of the well idle or fallow (2023) |
|`pct_cropland_500m_2023`		|	Percent land 500m of the well farmed, all crops (2023) |
|`crop_cycles_500m_2023`		|	Avg crops planted per year on farmland 500m of well (2023) |
|`pct_vineyard_1000m_2023`		|	Percent land 1km of the well w/ vineyards (2023) |
|`pct_grain_field_pasture_1000m_2023`	| Percent land 1km of the well w/ grain, hay, pasture (2023) |
|`pct_idle_fallow_1000m_2023`	|	Percent land 1km of the well idle or fallow (2023) |
|`pct_multicrop_1000m_2023`	|		Percent land 1km of the well planted 2+ times a year (2023) |
|`dist_veg_berry_m_2023`		|	Distance (m) from well to nearest veggie/berry field (2023) |
|`precip_hist_mean_annual_mm`	|	Mean annual rainfall from complete years before sample; in→mm. |
|`precip_prev_3fullmonths_mm`	|	SUM of 3 whole months before sample month. |
|`precip_prev_12fullmonths_mm`|		SUM of 12 whole months before sample month. |
|`temp_hist_mean_c`			|	Historic mean of months before sample; °F→°C.|
|`temp_prev_3fullmonths_c`	|		MEAN temperature of previous 3 complete months.|
|`temp_prev_12fullmonths_c`	|		MEAN temperature of previous 12 complete months.|